# La Mesa del Reino — Cortes automáticos

Convierte el episodio del domingo en 3–4 clips verticales con subtítulos,
listos para YouTube Shorts, Instagram y Facebook.

**Antes de empezar — obligatorio:**

`Entorno de ejecución` → `Cambiar tipo de entorno` → **GPU (T4)**

Sin GPU esto tarda horas en vez de minutos.

Motor: [opensource-clipping](https://github.com/NaufalRizqullah/opensource-clipping) (MIT).


## 1. Revisar la GPU


In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    print('SIN GPU.')
    print('Entorno de ejecucion -> Cambiar tipo de entorno -> GPU (T4), y vuelve a correr.')
    raise SystemExit('Falta GPU')
print('GPU lista:', r.stdout.strip())


## 2. Instalar (5–8 minutos, solo la primera vez)


In [ ]:
!git clone -q https://github.com/NaufalRizqullah/opensource-clipping.git /content/clip
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
%cd /content/clip
!pip install -q -r requirements.txt
print('Listo.')


## 3. Las llaves

En el panel izquierdo, el icono de la **llave (🔑 Secrets)**, agrega:

| Nombre | Para qué | ¿Obligatoria? |
|---|---|---|
| `GOOGLE_API_KEY` | Gemini elige los mejores momentos | **Sí** |
| `HF_TOKEN` | Pantalla dividida por orador | Opcional |

`GOOGLE_API_KEY` es la misma llave de Gemini del bot.

Para `HF_TOKEN`: crea el token en huggingface.co/settings/tokens **y acepta**
el acuerdo en huggingface.co/pyannote/speaker-diarization-3.1 — sin aceptarlo
el token no sirve.


In [ ]:
from google.colab import userdata
from pathlib import Path

def _get(name):
    try: return userdata.get(name) or ''
    except Exception: return ''

GOOGLE_API_KEY = _get('GOOGLE_API_KEY')
HF_TOKEN       = _get('HF_TOKEN')

if not GOOGLE_API_KEY:
    raise SystemExit('Falta GOOGLE_API_KEY en Secrets (icono de la llave).')

Path('/content/clip/.env').write_text(
    f'GOOGLE_API_KEY={GOOGLE_API_KEY}
HF_TOKEN={HF_TOKEN}
', encoding='utf-8')

SPLIT = bool(HF_TOKEN)
print('Gemini: ok')
print('Pantalla dividida:', 'ON' if SPLIT else 'OFF (sin HF_TOKEN)')


## 4. Generar los clips

Pega el link del episodio y ejecuta. Un episodio de 45 minutos toma
aproximadamente 20–40 minutos en una T4.


In [ ]:
URL   = 'https://www.youtube.com/watch?v=REEMPLAZA_ESTO'
CLIPS = 4

# gemini-2.5-flash (el respaldo que trae el proyecto) ya no existe para
# llaves nuevas: si el modelo principal falla, la corrida muere. Por eso
# fijamos un respaldo que si funciona.
args = [
    'python main.py',
    f'--url "{URL}"',
    f'--clips {CLIPS}',
    '--ratio "9:16"',
    '--words-per-sub 4',
    '--silence-trim',
    '--gemini-model gemini-3-flash-preview',
    '--gemini-fallback-model gemini-3.6-flash',
]
if SPLIT:
    args += ['--split-screen', '--dynamic-split']

cmd = 'cd /content/clip && ' + ' '.join(args)
print(cmd, '
')
!{cmd}


## 5. Descargar

Empaqueta todo en un zip y lo baja a tu computadora.


In [ ]:
import glob, os
from google.colab import files

vids = [p for p in glob.glob('/content/clip/outputs/**/*.mp4', recursive=True)
        if os.path.getsize(p) > 0]
if not vids:
    print('No se generaron clips. Revisa el error de la celda anterior.')
else:
    for p in sorted(vids):
        print(f'  {os.path.basename(p):<50} {os.path.getsize(p)/1048576:.1f} MB')
    !cd /content/clip && zip -qr /content/clips.zip outputs
    files.download('/content/clips.zip')


---

### Notas

- **Los títulos son de Richard.** Lo que Gemini sugiera es un borrador;
  el título que sale al aire lo decide él.
- **Revisa los subtítulos antes de publicar.** Whisper se equivoca con
  nombres propios y con términos bíblicos.
- `--words-per-sub 4` porque las palabras en español son largas; con 5 se
  desbordan en vertical.
- Para ir más rápido y con menos precisión, agrega `--use-dlp-subs`: usa los
  subtítulos que YouTube ya generó en vez de transcribir de nuevo.
- Colab desconecta si dejas la pestaña inactiva mucho rato.
